In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from datasets import load_dataset
import json
import logging

# 配置日志记录
logging.basicConfig(
    filename='inference_log.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# 配置模型和数据集路径
MODEL_PATH = 'Qwen/Qwen2.5-Math-1.5B-Instruct'
TEST_DATASET_PATH = "Finetune_data\test_finetune.json"
LORA_PATH1 = 'Finetune_LLM\qwen2.5_math_Ent\checkpoint-8000'
LORA_PATH2 = 'Finetune_LLM\qwen2.5_math_QA\checkpoint-24000'


# 创建知识库，添加pos和z张量
data = torch.load('qm9_dataset_small.pt')
knowledge_base = {}
for entry in data:
    smile = entry['smile']  # 使用smile作为索引
    pos = entry['pos']  # 使用pos作为值
    z = entry['z']  # 获取z张量
    knowledge_base[smile] = {'pos': pos, 'z': z}  # 存储pos和z张量

# 加载tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

# 加载模型
model1 = AutoModelForCausalLM.from_pretrained(MODEL_PATH, trust_remote_code=True)
model2 = AutoModelForCausalLM.from_pretrained(MODEL_PATH, trust_remote_code=True)

# 加载LoRA微调权重
model1 = PeftModel.from_pretrained(model1, model_id=LORA_PATH1)
model2 = PeftModel.from_pretrained(model2, model_id=LORA_PATH2)

# 将模型和数据传送到NPU设备
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model1.to(device)
model2.to(device)

# 数据集加载
test_dataset = load_dataset('json', data_files=TEST_DATASET_PATH)['train']

def get_smile(input_data):
    messages = [
        {"role": "system", "content": "Extract the organic molecule formula mentioned in the following words."},
        {"role": "user", "content": f"{input_data}"}
    ]

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # Tokenize and transfer to device
    model_inputs = tokenizer([text], return_tensors="pt", padding=True, truncation=True)

    # Transfer tensors to the device (GPU/CPU)
    model_inputs = {key: value.to(device) for key, value in model_inputs.items()}

    # 生成预测输出
    generated_ids = model1.generate(
        model_inputs['input_ids'],
        attention_mask=model_inputs['attention_mask'],  # Pass attention_mask explicitly
        max_new_tokens=50
    )
    
    generated_ids = [
        output_ids[len(model_inputs['input_ids'][0]):] for output_ids in generated_ids
    ]

    # 解码生成的文本
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response


# 检索函数：根据smile从知识库中检索pos和z张量
def retrieve_pos_and_z_from_knowledge_base(smile):
    entry = knowledge_base.get(smile)
    if entry:
        return entry['pos'], entry['z']
    return None, None

# 推理函数：基于模型生成输出
def generate_inference(input_data):
    # 获取smile并检索pos和z张量
    smile_query = get_smile(input_data['Instruction'])

    print('smile:', smile_query)
    pos_tensor, z_tensor = retrieve_pos_and_z_from_knowledge_base(smile_query)
    print('rag_pos:', pos_tensor, 'rag_z:', z_tensor)

    if pos_tensor is None or z_tensor is None:
        # 如果知识库中没有对应的分子，使用model2进行推理
        prompt = input_data['Instruction'] + input_data['Input']
        
        # 创建消息
        messages = [
            {"role": "system", "content": "You are a scientist to generate the position tensor and atomic number tensor."},
            {"role": "user", "content": f"{prompt}"}
        ]

        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

        # Tokenize and transfer to device
        model_inputs = tokenizer([text], return_tensors="pt", padding=True, truncation=True)

        # Transfer tensors to the device (GPU/CPU)
        model_inputs = {key: value.to(device) for key, value in model_inputs.items()}

        # 生成预测输出
        generated_ids = model2.generate(
            model_inputs['input_ids'],
            attention_mask=model_inputs['attention_mask'],  # Pass attention_mask explicitly
            max_new_tokens=1200
        )
        
        generated_ids = [
            output_ids[len(model_inputs['input_ids'][0]):] for output_ids in generated_ids
        ]

        # 解码生成的文本
        response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        return response
    else:
        # 如果知识库中找到了分子，直接返回对应的pos和z张量
        return f"pos tensor: {pos_tensor}\nz tensor: {z_tensor}"

# 收集生成的文本和实际的输出文本
generated_outputs = []
actual_outputs = []
input_data_list = []

for example in test_dataset:
    generated_text = generate_inference(example)
    generated_outputs.append(generated_text)
    print('generate:', generated_text)
    actual_outputs.append(example['Output'])
    input_data_list.append({
        'Instruction': example['Instruction'],
        'Input': example['Input'],
        'Generated Output': generated_text,
        'Actual Output': example['Output']
    })

# 保存推理输入和输出到文件
with open('generate_output.json', 'w') as f:
    json.dump(input_data_list, f, indent=4)


smile: N#CC#N
rag_pos: tensor([[ 1.7990e-02, -1.1564e+00, -4.7930e-03],
        [ 1.8490e-03, -3.1390e-03,  2.6790e-03],
        [-1.6535e-02,  1.3719e+00,  9.5490e-03],
        [-3.2072e-02,  2.5252e+00,  1.5650e-02]]) rag_z: tensor([7, 6, 6, 7])
generate: pos tensor: tensor([[ 1.7990e-02, -1.1564e+00, -4.7930e-03],
        [ 1.8490e-03, -3.1390e-03,  2.6790e-03],
        [-1.6535e-02,  1.3719e+00,  9.5490e-03],
        [-3.2072e-02,  2.5252e+00,  1.5650e-02]])
z tensor: tensor([7, 6, 6, 7])
smile: OC1CC2(CO2)C1
rag_pos: tensor([[ 0.6071,  0.8594, -0.6558],
        [-0.0348, -0.1750,  0.0555],
        [ 0.0990, -0.2324,  1.6049],
        [-1.3788, -0.6160,  1.6456],
        [-2.0302, -1.7080,  2.3651],
        [-2.2600, -0.3307,  2.7216],
        [-1.5836, -0.1506,  0.2054],
        [ 0.3380,  1.7070, -0.2828],
        [ 0.3064, -1.0995, -0.4087],
        [ 0.8301, -0.9166,  2.0350],
        [ 0.2481,  0.7663,  2.0253],
        [-2.8806, -2.2168,  1.9190],
        [-1.4677, -2.2852,  